In [1]:
from langchain_openai import ChatOpenAI

In [2]:
from langchain.agents import create_agent

In [3]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage

In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.tools import load_mcp_tools

In [5]:
from langchain.tools import tool

In [50]:
from typing import List, Union, Optional, Dict, Literal, Callable

In [7]:
import tiktoken

In [8]:
import os
from dotenv import load_dotenv

In [9]:
load_dotenv()

True

In [10]:
from pathlib import Path

In [22]:
import asyncio

In [11]:
llm = ChatOpenAI(
    base_url= os.getenv("BASE_URL"),
    api_key=os.getenv("API_KEY"),
    model=os.getenv("LLM"),
)

In [12]:
# @tool("calculator", description="Performs arithmetic calculations. Use this for any math problems.")
# def calculator(expression: str) -> str:
#     """Evaluate mathematical expressions."""
#     return str(eval(expression))

In [13]:
client = MultiServerMCPClient(
    {
        "webgent": {
            "transport": "stdio",
            "command": "uv",
            "args": [
                "run",
                "--directory",
                "/home/butcher/projects/webgent-mcp",
                "webgent",
            ],
            "env": {
                "WEBGENT_TRANSPORT": "stdio",
                "WEBGENT_HEADLESS": "false",
            },
        }
    }
)

In [52]:
class Server:
    
    def __init__(
        self,
        name: str,
        transport: Literal["stdio", "http"] = "http",
        url: Optional[str] = None,
        command: Optional[str] = None,
        args: Optional[List[str]] = None,
        env: Optional[Dict[str, Any]] = None
    ):
        
        self.name = name
        self.transport = transport
        self.url = url
        self.command = command
        self.args = args
        self.env = env

    def dump_json(self):
        config = {}
        config["transport"] = self.transport
        if self.env is not None:
            config["env"] = self.env
        
        match self.transport:
            case "http":
                config["url"] = self.url
            case "stdio":
                config["command"] = self.command
                config["args"] = self.args

        return config  

In [24]:
class DuplicateValueError(Exception):
    """Exception raised when a duplicate value is detected."""
    pass

In [37]:
class MCPClient:
    
    def __init__(
        self,
        servers: List[Server]
    ):
        self.servers = servers
        self.names = [server.name for server in servers]
        self.client = None

    def connect(self):
        servers = {}
        for server in self.servers:
            if server.name in servers:
                raise DuplicateValueError(f"server name {server.name} found twice")
            servers[server.name] = server.dump_json()

        self.client = MultiServerMCPClient(servers)

    def append(self, server: Server):
        if server.name in self.names:
            raise DuplicateValueError(f"server with name {server.name} already there")
        self.server.append(server)
        return True

    async def get_tools(self):
        if self.client is None:
            self.connect()

        return await self.client.get_tools()

        

In [38]:
server = Server(
    name="myserver",
    transport="http",
    url="http://127.0.0.1:8000/mcp"
)

In [39]:
client = MCPClient(
    servers=[server]
)

In [40]:
mcp_tools = await client.get_tools()

In [41]:
# tools = [calculator]
# tool_map = {tool.name: tool for tool in tools}

In [42]:
len(mcp_tools)

2

In [43]:
mcp_tools

[StructuredTool(name='execute_command', args_schema={'additionalProperties': False, 'properties': {'command': {'type': 'string'}}, 'required': ['command'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, handle_tool_error=<function _handle_mcp_tool_error at 0x7f604d3bb950>, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7f60466da1f0>),
 StructuredTool(name='read_file', args_schema={'additionalProperties': False, 'properties': {'path': {'type': 'string'}}, 'required': ['path'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, handle_tool_error=<function _handle_mcp_tool_error at 0x7f604d3bb950>, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7f60466e7e20>)]

In [16]:
import json

In [17]:
class InternalTools:
    @classmethod
    def tools(cls):
        @tool("collapsed_tool_result", description="Fetch old collapsed tool result using tool call id.")
        def collapsed_tool_result(tool_call_id: str) -> str:
            # Ensure path is a Path object and join it with the filename
            file_path = Thread.get_tool_result_path() / tool_call_id
            
            try:
                # Direct, clean reading using pathlib
                return file_path.read_text(encoding="utf-8")
            except Exception as e:
                return str(e)

        return [collapsed_tool_result]


In [18]:
from langchain_openai import ChatOpenAI
from typing import List, Iterator, Any

class Agent:
    def __init__(self, model: ChatOpenAI, tools=List):
        self.model = model
        self.tools = tools
        if self.tools:
            self.tools.extend(InternalTools.tools())
            self.model = self.model.bind_tools(self.tools)
        self.tool_map = {tool.name: tool for tool in self.tools}
            
    async def ainvoke(self, thread: Thread, self_append: bool = True):
        if thread.tail is not None:
            thread = thread.tail
        thread.agent = self
        while True:
            response = await self.model.ainvoke(thread.messages)
            
            if not self_append:
                thread.agent = None
                return response
                
            thread.append(response)
            if response.tool_calls:
                for tool in response.tool_calls:
                    args=tool["args"]
                    call_id = tool["id"]
                    name = tool["name"]
                    result = await self.tool_map[name].ainvoke(args)
                    thread.append(ToolMessage(name=name, content=str(result), tool_call_id=call_id))
            else:
                thread.agent = None
                return response

    def invoke(self, thread: Thread, self_append: bool = True):
        import asyncio

        return asyncio.run(
            self.ainvoke(
                thread,
                self_append=self_append,
            )
        )
    
    def stream(
        self,
        thread: Thread,
        self_append: bool = True,
    ) -> Iterator[Any]:

        if thread.tail is not None:
            thread = thread.tail

        thread.agent = self

        try:
            while True:

                # Accumulated complete AI response
                response = None

                # Stream from model
                for chunk in self.model.stream(thread.messages):

                    # Send chunk to caller immediately
                    yield chunk

                    # Accumulate chunks
                    if response is None:
                        response = chunk
                    else:
                        response = response + chunk

                # If requested, save complete response
                if not self_append:
                    return

                thread.append(response)

                # Check for tool calls after the complete
                # streamed response has been assembled
                if response.tool_calls:

                    for tool in response.tool_calls:

                        args = tool["args"]
                        call_id = tool["id"]
                        name = tool["name"]

                        result = self.tool_map[name].invoke(args)

                        tool_message = ToolMessage(
                            name=name,
                            content=result,
                            tool_call_id=call_id,
                        )

                        thread.append(tool_message)

                        # Continue while-loop.
                        # The next iteration sends:
                        #
                        # previous messages
                        # + AIMessage(tool_call)
                        # + ToolMessage(result)
                        #
                        # back to the model.

                else:
                    return

        finally:
            thread.agent = None


    def __ror__(self, thread: Thread):
        return self.invoke(thread)

In [19]:
class ThreadHideRule:
    def __init__(
        self,
        name: str,
        message: str,
    ):
        self.name = name
        self.message = message

In [20]:
class AutoToolHideRule:
    def __init__(
        self,
        token_limit: int,
        per_tool_token_limit: Optional[int] = None
    ):
        self.token_limit = token_limit
        self.per_tool_token_limit = per_tool_token_limit

In [21]:
class Thread:
    def __init__(
        self, 
        messages: List[Union[AIMessage, HumanMessage, ToolMessage, SystemMessage]] = None, 
        system_prompt: Union[str, SystemMessage] = None,
        compression_prompt: str = None,
        token_limit: int = None,
        tool_hide_rules: List[Union[ThreadHideRule, AutoToolHideRule]] = None   
    ):
        self.messages = []
        self.system_prompt = system_prompt
        self.compression_prompt = compression_prompt
        self.token_limit = token_limit
        self.agent = None
        self.encoder = tiktoken.encoding_for_model("gpt-4o-mini")
        self.root = None
        self.parent = None
        self.child = None
        self.tail = None
        self.tool_hide_rules = tool_hide_rules
        self.path = Path.cwd() / "tool_results"
        self.path.mkdir(parents=True, exist_ok=True)

        
        if messages is not None:
            index = self._find_system_message(messages)
            if index == -1 or index == 0:
                self.messages = messages
            else:
                raise Valueerror(
                    f"system message not at the starting, it was found at {index} index"
                )
                    
        if self.system_prompt is not None:
            index = self._find_system_message(self.messages)
            if isinstance(self.system_prompt, str):
                self.system_prompt = SystemMessage(self.system_prompt)
            if index == 0:
                if len(self.messages) == 0:
                    self.append(self.system_prompt)
                else:
                    self[0] = self.system_prompt
            elif index == -1:
                self.messages = [self.system_prompt] + self.messages
            else:
                pass

    @classmethod
    def get_tool_result_path(cls):
        path = Path.cwd() / "tool_results"
        path.mkdir(parents=True, exist_ok=True)
        return path

    def _find_system_message(self, messages: List[Union[AIMessage, HumanMessage, ToolMessage, SystemMessage]]):
        for i in range(len(messages)):
            if isinstance(messages[i], SystemMessage):
                return i
        return -1

    def count_token(self):
        if self.tail is not None:
            content = [m.content for m in self.tail]
        else:
            content = [m.content for m in self]
        merged = "\n".join(content)
        return len(self.encoder.encode(merged))

    def calculate_tokens(self, content):
        return len(self.encoder.encode(content))
        
        
    def append(self, message: Union[AIMessage, HumanMessage, ToolMessage, SystemMessage]):
        if self.root is not None:
            tool_hide_rules = self.root.tool_hide_rules
        else:
            tool_hide_rules = self.tool_hide_rules

        thread_hide_rules = [rule for rule in tool_hide_rules if isinstance(rule, ThreadHideRule)]
        
        auto_tool_hide_rules = None
        for rule in tool_hide_rules:
            if isinstance(rule, AutoToolHideRule):
                auto_tool_hide_rules = rule
                break
        
            
        if isinstance(message, ToolMessage) and tool_hide_rules is not None:
            name = message.name
            match = False
            tool_hide_rule = None
            for rule in thread_hide_rules:
                if rule.name == name:
                    match = True
                    tool_hide_rule = rule
                    break
            if match:
                for m in reversed(self):
                    if isinstance(m, ToolMessage) and m.name == name:
                        self.save_tool_result(m)
                        m.content = tool_hide_rule.message + f"\n tool call id {m.tool_call_id}. use the ID to retrive this tool result using collapsed_tool_result"
                        break
                        
        if auto_tool_hide_rules is not None:
            total_tokens = self.count_token()
            if total_tokens >= auto_tool_hide_rules.token_limit:
                if auto_tool_hide_rules.per_tool_token_limit is not None:
                    per_tool_token_limit = auto_tool_hide_rules.per_tool_token_limit
                    for m in reversed(self):
                        if isinstance(m, ToolMessage) and self.calculate_tokens(m.content) > per_tool_token_limit:
                            self.save_tool_result(m)
                            m.content = f"This tool call result has been collapsed due to token size constraints.\n tool call id {m.tool_call_id}. use the ID to retrive this tool result using collapsed_tool_result"
                else:
                    per_tool_token_limit = 8_000
                    for m in reversed(self):
                        if isinstance(m, ToolMessage) and self.calculate_tokens(m.content) > per_tool_token_limit:
                            self.save_tool_result(m)
                            m.content = f"This tool call result has been collapsed due to token size constraints.\n tool call id {m.tool_call_id}. use the ID to retrive this tool result using collapsed_tool_result"

            
        

                        
        token_usuage = self.count_token()
        if self.token_limit is not None and token_usuage > self.token_limit and self.agent is not None and self.compression_prompt is not None:
            self.messages.append(HumanMessage(self.compression_prompt))
            compression_report = self.agent.invoke(self, self_append=False).content
            self.messages.pop()
            
            new_thread = self.copy()
            self.child = new_thread
            new_thread.parent = self
            new_thread.root = self.root if self.root is not None else self
            self.root.tail = new_thread

            new_thread.messages = []
            if isinstance(self.root[0], SystemMessage):
                new_thread.append(self.root[0])
            new_thread.append(HumanMessage(compression_report)) 
        else:
            self.messages.append(message)

    def save_tool_result(self, message: ToolMessage) -> bool:
        try:
            with open(self.path / str(message.tool_call_id), "w", encoding="utf-8") as f:
                f.write(message.content)
            return True
        except:
            return False

    def count(self):
        counts = {
            "depth":0,
            "system":0, 
            "human":0,
            "ai":0,
            "tool":0
        }
        thread = self
        depth = 0
        if isinstance(thread[0], SystemMessage):
            counts["system"]=1
        while True:
            for m in thread:
                if isinstance(m, AIMessage):
                    counts["ai"]+=1
                elif isinstance(m, HumanMessage):
                    counts["human"]+=1
                elif isinstance(m, ToolMessage):
                    counts["tool"]+=1
                else:
                    pass
            if thread.child is None:
                break
            else:
                thread = thread.child
                depth += 1
        counts["depth"] = depth
        return counts

    def __ror__(self, other: Union[AIMessage, HumanMessage, ToolMessage, SystemMessage]):
        if self.tail is not None:
            self.tail.append(other)
        else:
            self.append(other)

    # def __add__(self, other: Thread):
    #     new_thread = Thread()
    #     new_thread.messages = self.messages + other.messages
    #     return new_thread

    def __str__(self):
        counts = self.count()
        return json.dumps(counts)

    def __repr__(self):
        counts = self.count()
        return json.dumps(counts)

    def __iter__(self):
        if self.tail is not None:
            for msg in self.tail.messages:
                yield msg
        else:
            for msg in self.messages:
                yield msg
                
    def __getitem__(self, index):
        if self.tail is not None:
            return self.tail.messages[index]
        else:
            return self.messages[index]

    def __len__(self):
        if self.tail is not None:
            return len(self.tail.messages)
        else:
            return len(self.messages)

    def __setitem__(self, index, value):
        if self.tail is not None:
            self.tail.messages[index] = value
        else:
            self.messages[index] = value

    def __copy__(self):
        new_instance = Thread()
        return new_instance
        

In [38]:
session_cm = client.session("webgent")
session = await session_cm.__aenter__()

mcp_tools = await load_mcp_tools(session)

In [39]:
myagent = Agent(
    model=llm,
    tools=mcp_tools
)

In [40]:
thread = Thread(
    tool_hide_rules=[
         AutoToolHideRule(
             token_limit = 20_000
         )
    ]
)

In [41]:
thread.append(SystemMessage("You are a web agent"))

In [46]:
HumanMessage("go to amazon") | thread

In [47]:
thread.messages

[SystemMessage(content='You are a web agent', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='open browser session', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 84, 'prompt_tokens': 5442, 'total_tokens': 5526, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Deepseek-vapt', 'system_fingerprint': 'vllm-0.25.0-tp2-ep-d4f8ac0c', 'id': 'chatcmpl-99927e77e9ceb109', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a06620-a585-70d1-9379-f0783409a4f7-0', tool_calls=[{'name': 'create_session', 'args': {'profile_name': 'default', 'initial_url': 'about:blank'}, 'id': 'chatcmpl-tool-9b1ecceeb5b49af8', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 5442, 'output_tokens': 84, 'total_tokens': 5526, 'input_token_details': {}, 'output_token_details': {}}),
 ToolM

In [48]:
# for chunk in myagent.stream(thread):
#     print(chunk.content, end="", flush=True)

In [49]:
response = await myagent.ainvoke(thread)

In [37]:
async with client.session("webgent") as session:
    mcp_tools = await load_mcp_tools(session)

    myagent = Agent(
        model=llm,
        tools=mcp_tools
    )

    response = await myagent.ainvoke(thread)

In [37]:
response

AIMessage(content="Hello again! 😊\n\nIs there anything you'd like me to help you with on the web today? Maybe:\n\n- Browse a website or search for something?\n- Fill out a form or log into a service?\n- Scrape some data from a page?\n- Test something on a site?\n\nJust let me know what you need and I'll get right to it!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 115, 'prompt_tokens': 5487, 'total_tokens': 5602, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Deepseek-vapt', 'system_fingerprint': 'vllm-0.25.0-tp2-ep-d4f8ac0c', 'id': 'chatcmpl-947a944e2629dc75', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a065f9-35b2-77f3-98b0-97c2f8227727-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 5487, 'output_tokens': 115, 'total_tokens': 5602, 'input_token_details': {}, 'output_token_details': {}})

In [31]:
response.content

'Hi there! How can I help you today? If you need me to browse a website, fill out a form, or do anything else on the web, just let me know!'

In [29]:
thread.messages

[SystemMessage(content='You are a web agent', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='go to amazon.in', additional_kwargs={}, response_metadata={})]

In [39]:
for m in response:
    if isinstance(m, SystemMessage):
        print("system message:\n")
    elif isinstance(m, HumanMessage):
        print("human message:\n")
    elif isinstance(m, AIMessage):
        print("ai message:\n")
    elif isinstance(m, ToolMessage):
        print("tool message:\n")
    else:
        print("other type:\n")
    print(m.model_dump_json())
    print("\n\n")

system message:

{"content":"You are a Mathematical Expression solver","additional_kwargs":{},"response_metadata":{},"type":"system","name":null,"id":null}



human message:

{"content":"Solve this particular expression : 3*4+77 use calculator tool","additional_kwargs":{},"response_metadata":{},"type":"human","name":null,"id":null}



ai message:

{"content":"","additional_kwargs":{"refusal":null},"response_metadata":{"token_usage":{"completion_tokens":70,"prompt_tokens":297,"total_tokens":367,"completion_tokens_details":null,"prompt_tokens_details":null},"model_provider":"openai","model_name":"Deepseek-vapt","system_fingerprint":"vllm-0.25.0-tp2-ep-d4f8ac0c","id":"chatcmpl-bc18df2e93c1cfe1","finish_reason":"tool_calls","logprobs":null},"type":"ai","name":null,"id":"lc_run--01a03256-a5d0-7433-8fa3-9299a904fa75-0","tool_calls":[{"name":"calculator","args":{"expression":"3*4+77"},"id":"chatcmpl-tool-8e2f16389057886f","type":"tool_call"}],"invalid_tool_calls":[],"usage_metadata":{"input_t

In [19]:
s = SystemMessage("hii")

In [20]:
s.model_dump_json()

'{"content":"hii","additional_kwargs":{},"response_metadata":{},"type":"system","name":null,"id":null}'

In [45]:
response["messages"][-1].content

'The result of the expression \\(3 \\times 4 + 77\\) is **89**.'

In [47]:
response["messages"].append(HumanMessage("Now also calculate this one 456%100"))

In [54]:
nr = agent.invoke(nr)

In [74]:
tool = mcp_tools[0]

print(tool.name)
print(tool.args_schema)

create_session
{'additionalProperties': False, 'properties': {'profile_name': {'default': 'default', 'type': 'string'}, 'initial_url': {'default': 'about:blank', 'type': 'string'}}, 'type': 'object'}


In [75]:
result = await tool.ainvoke({
    "profile_name": "default",
    "initial_url": "about:blank",
})

print(result)

[{'type': 'text', 'text': '{"session_id":"ba7b1c0e-7c32-4842-a75e-f8f91e6d5635","profile_name":"default","page_id":"b500c32e-d571-417d-bc59-821bc5cdb5a7","url":"about:blank","title":""}', 'id': 'lc_12b1d9f4-209b-4c4b-8ad2-bb751f60144e'}]


In [76]:
type(result)

list